In [15]:
import pymupdf
from langchain_core.documents import Document
from transformers import CLIPProcessor , CLIPModel
from PIL import Image 
import torch
import numpy as np
from langchain.chat_models import init_chat_model
from langchain_core.prompts import PromptTemplate
from langchain_classic.schema  import HumanMessage
from sklearn.metrics.pairwise import cosine_similarity
import os
import base64
from langchain_classic.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
import io


In [16]:
import os
from dotenv import load_dotenv
from transformers import CLIPProcessor, CLIPModel

load_dotenv()

# Groq API key
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

# Initialize CLIP
model_name = "openai/clip-vit-base-patch32"

clip_model = CLIPModel.from_pretrained(model_name)
clip_processor = CLIPProcessor.from_pretrained(model_name)

clip_model.eval()

print("CLIP loaded successfully!")

Loading weights: 100%|██████████| 398/398 [00:00<00:00, 15771.34it/s]


CLIP loaded successfully!


In [20]:
## embedding function for image
def embed_image(image_data):
    """Embed image using CLIP"""

    if isinstance(image_data, str):
        image = Image.open(image_data).convert("RGB")
    else:
        image = image_data

    inputs = clip_processor(
        images=image,
        return_tensors="pt"
    )

    with torch.no_grad():
        outputs = clip_model.get_image_features(**inputs)

        # Extract the actual tensor
        if hasattr(outputs, "pooler_output"):
            features = outputs.pooler_output
        else:
            features = outputs

        # Normalize
        features = features / features.norm(
            dim=-1,
            keepdim=True
        )

        return features.squeeze().cpu().numpy()

## embedding function for text

def embed_text(text):
    """Embed text using CLIP"""

    inputs = clip_processor(
        text=text,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=77
    )

    with torch.no_grad():
        outputs = clip_model.get_text_features(**inputs)

        # Extract the actual tensor
        if hasattr(outputs, "pooler_output"):
            features = outputs.pooler_output
        else:
            features = outputs

        # Normalize
        features = features / features.norm(
            dim=-1,
            keepdim=True
        )

        return features.squeeze().cpu().numpy()

In [21]:
## process the pdf
pdf_path = "multimodal_sample.pdf"
doc = pymupdf.open(pdf_path)
## storage for all documents embeddings
all_doc =[]
all_embeddings=[]
image_data = {}
# text splitter

text_splitter = RecursiveCharacterTextSplitter(chunk_size = 500 , chunk_overlap = 100)

doc

Document('multimodal_sample.pdf')

In [22]:
# Processing text and images page by page

for i, page in enumerate(doc):

    # ==========================================
    # PROCESS TEXT
    # ==========================================

    text = page.get_text()

    if text.strip():

        temp_doc = Document(
            page_content=text,
            metadata={
                "page": i,
                "type": "text"
            }
        )

        text_chunks = text_splitter.split_documents(
            [temp_doc]
        )

        # Embed each text chunk using CLIP
        for chunk in text_chunks:

            embedding = embed_text(
                chunk.page_content
            )

            all_embeddings.append(embedding)
            all_doc.append(chunk)

    # ==========================================
    # PROCESS IMAGES
    # ==========================================

    for img_index, img in enumerate(
        page.get_images(full=True)
    ):

        try:

            # Get image reference
            xref = img[0]

            # Extract image from PDF
            base_image = doc.extract_image(xref)

            image_bytes = base_image["image"]

            # Convert bytes → PIL Image
            pil_image = Image.open(
                io.BytesIO(image_bytes)
            ).convert("RGB")

            # Create unique image ID
            image_id = f"page_{i}_img_{img_index}"

            # ==================================
            # STORE IMAGE AS BASE64
            # ==================================

            buffered = io.BytesIO()

            pil_image.save(
                buffered,
                format="PNG"
            )

            img_base64 = base64.b64encode(
                buffered.getvalue()
            ).decode()

            image_data[image_id] = img_base64

            # ==================================
            # EMBED IMAGE USING CLIP
            # ==================================

            embedding = embed_image(
                pil_image
            )

            all_embeddings.append(embedding)

            # ==================================
            # CREATE DOCUMENT
            # ==================================

            image_doc = Document(
                page_content=f"[Image:{image_id}]",
                metadata={
                    "page": i,
                    "type": "image",
                    "image_id": image_id
                }
            )

            all_doc.append(image_doc)

        except Exception as e:

            print(
                f"Error processing image "
                f"{img_index} on page {i}: {e}"
            )

            continue


# Close PDF
doc.close()